# YOLOv8n + CBAM 注意力模块 — RSNA 肺炎检测实验

## 改进说明
- **CBAM** (Convolutional Block Attention Module) 同时建模通道注意力和空间注意力
- 仅在 backbone 的 P3/P4/P5 层后添加 CBAM，不改变 head 结构
- 使用实际 YOLOv8n 通道数，确保与预训练权重兼容

In [ ]:
import torch
print("CUDA是否可用:", torch.cuda.is_available())
print("GPU型号:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无GPU，请检查运行时设置")
print("PyTorch版本:", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/Medical_project/rsna-yolov8"
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
YOLO_DATA_DIR = os.path.join(DATASET_DIR, "rsna_yolo")

for dir_path in [PROJECT_ROOT, DATASET_DIR, YOLO_DATA_DIR]:
    os.makedirs(dir_path, exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATA_DIR, "images/train"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATA_DIR, "images/val"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATA_DIR, "labels/train"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATA_DIR, "labels/val"), exist_ok=True)

print("项目目录初始化完成，根目录路径:", PROJECT_ROOT)

In [ ]:
# 升级pip并安装依赖
!pip install --upgrade pip
!pip install ultralytics pydicom pandas opencv-python numpy

# 验证安装成功
!yolo version

In [ ]:
from google.colab import files
import os

print("请上传你下载的kaggle.json文件")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions list

In [ ]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/Medical_project/rsna-yolov8"
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
RAW_DATA_DIR = os.path.join(DATASET_DIR, "rsna-pneumonia-detection-challenge")

%cd $DATASET_DIR
!kaggle competitions download -c rsna-pneumonia-detection-challenge

!unzip -q rsna-pneumonia-detection-challenge.zip -d $RAW_DATA_DIR

print("数据集解压完成，文件列表：")
!ls $RAW_DATA_DIR

In [ ]:
%%writefile /content/drive/MyDrive/Medical_project/rsna-yolov8/rsna_to_yolo.py
import os
import pydicom
import pandas as pd
import cv2
import numpy as np
from tqdm import tqdm

RAW_DATA_DIR = "/content/drive/MyDrive/Medical_project/rsna-yolov8/dataset/rsna-pneumonia-detection-challenge"
YOLO_DATA_DIR = "/content/drive/MyDrive/Medical_project/rsna-yolov8/dataset/rsna_yolo"
IMAGE_SIZE = 512

labels_df = pd.read_csv(os.path.join(RAW_DATA_DIR, "stage_2_train_labels.csv"))
image_ids = labels_df["patientId"].unique()

print(f"总共有 {len(image_ids)} 张图像，开始转换...")

for patient_id in tqdm(image_ids):
    dicom_path = os.path.join(RAW_DATA_DIR, "stage_2_train_images", f"{patient_id}.dcm")
    dicom = pydicom.dcmread(dicom_path)
    image = dicom.pixel_array

    orig_h, orig_w = image.shape
    scale_w = IMAGE_SIZE / orig_w
    scale_h = IMAGE_SIZE / orig_h
    image_resized = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))

    img_save_path = os.path.join(YOLO_DATA_DIR, "images/train", f"{patient_id}.jpg")
    cv2.imwrite(img_save_path, image_resized)

    label_save_path = os.path.join(YOLO_DATA_DIR, "labels/train", f"{patient_id}.txt")
    box_df = labels_df[labels_df["patientId"] == patient_id]

    with open(label_save_path, "w") as f:
        for _, row in box_df.iterrows():
            if row["Target"] == 0:
                continue
            x = row["x"] * scale_w
            y = row["y"] * scale_h
            w = row["width"] * scale_w
            h = row["height"] * scale_h

            x_center = (x + w / 2) / IMAGE_SIZE
            y_center = (y + h / 2) / IMAGE_SIZE
            width_norm = w / IMAGE_SIZE
            height_norm = h / IMAGE_SIZE

            f.write(f"0 {x_center:.6f} {y_center:.6f} {width_norm:.6f} {height_norm:.6f}\n")

print("数据转换完成！")

In [ ]:
%cd /content/drive/MyDrive/Medical_project/rsna-yolov8
!python rsna_to_yolo.py

In [ ]:
import os
import random
import shutil
from tqdm import tqdm

YOLO_DATA_DIR = "/content/drive/MyDrive/Medical_project/rsna-yolov8/dataset/rsna_yolo"
train_img_dir = os.path.join(YOLO_DATA_DIR, "images/train")
val_img_dir = os.path.join(YOLO_DATA_DIR, "images/val")
train_label_dir = os.path.join(YOLO_DATA_DIR, "labels/train")
val_label_dir = os.path.join(YOLO_DATA_DIR, "labels/val")

random.seed(42)
all_images = [f for f in os.listdir(train_img_dir) if f.endswith(".jpg")]
random.shuffle(all_images)

val_ratio = 0.2
val_num = int(len(all_images) * val_ratio)
val_images = all_images[:val_num]

print(f"总图像数：{len(all_images)}，训练集：{len(all_images)-val_num}，验证集：{val_num}")

for img_name in tqdm(val_images):
    src_img = os.path.join(train_img_dir, img_name)
    dst_img = os.path.join(val_img_dir, img_name)
    shutil.move(src_img, dst_img)

    label_name = img_name.replace(".jpg", ".txt")
    src_label = os.path.join(train_label_dir, label_name)
    dst_label = os.path.join(val_label_dir, label_name)
    if os.path.exists(src_label):
        shutil.move(src_label, dst_label)

print("训练集/验证集划分完成！")

In [ ]:
%%writefile /content/drive/MyDrive/Medical_project/rsna-yolov8/dataset/rsna_yolo/data.yaml
path: /content/drive/MyDrive/Medical_project/rsna-yolov8/dataset/rsna_yolo
train: images/train
val: images/val
nc: 1
names:
  0: nodule

## CBAM 模型定义与训练

以下为 CBAM 改进部分：定义 CBAM 模块 → 生成自定义 YAML → 训练模型

In [ ]:
# 定义 CBAM 模块并注册到 ultralytics 解析器
import torch
import torch.nn as nn
from ultralytics import YOLO


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        mid = max(channels // reduction, 1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.SiLU(),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(combined))


class CBAM(nn.Module):
    """CBAM = Channel Attention + Spatial Attention"""
    def __init__(self, c1, c2=None, reduction=16, kernel_size=7):
        super().__init__()
        c2 = c2 or c1
        self.ca = ChannelAttention(c1, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


# 注册 CBAM 到 ultralytics 解析器
import ultralytics.nn.tasks as tasks
tasks.parse_model.__globals__['CBAM'] = CBAM

print("CBAM 模块已定义并注册到 ultralytics 解析器")

In [ ]:
# 生成带 CBAM 的 YOLOv8n YAML 配置
# YOLOv8n 实际通道数: 16, 32, 64, 128, 256
import os

PROJECT_ROOT = "/content/drive/MyDrive/Medical_project/rsna-yolov8"
YAML_PATH = os.path.join(PROJECT_ROOT, "yolov8_cbam.yaml")

yaml_content = """nc: 1
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [16, 3, 2]]           #  0  P1/2
  - [-1, 1, Conv, [32, 3, 2]]           #  1  P2/4
  - [-1, 1, C2f, [32, True]]            #  2
  - [-1, 1, Conv, [64, 3, 2]]           #  3  P3/8
  - [-1, 2, C2f, [64, True]]            #  4
  - [-1, 1, CBAM, [64]]                 #  5  CBAM at P3
  - [-1, 1, Conv, [128, 3, 2]]          #  6  P4/16
  - [-1, 2, C2f, [128, True]]           #  7
  - [-1, 1, CBAM, [128]]                #  8  CBAM at P4
  - [-1, 1, Conv, [256, 3, 2]]          #  9  P5/32
  - [-1, 1, C2f, [256, True]]           # 10
  - [-1, 1, SPPF, [256, 5]]             # 11
  - [-1, 1, CBAM, [256]]                # 12  CBAM at P5 (after SPPF)

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 13
  - [[-1, 8], 1, Concat, [1]]                     # 14  cat P4 (CBAM output)
  - [-1, 1, C2f, [128]]                           # 15
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 16
  - [[-1, 5], 1, Concat, [1]]                     # 17  cat P3 (CBAM output)
  - [-1, 1, C2f, [64]]                            # 18  P3/8-small
  - [-1, 1, Conv, [64, 3, 2]]                     # 19
  - [[-1, 15], 1, Concat, [1]]                    # 20  cat head P4
  - [-1, 1, C2f, [128]]                           # 21  P4/16-medium
  - [-1, 1, Conv, [128, 3, 2]]                    # 22
  - [[-1, 12], 1, Concat, [1]]                    # 23  cat P5 (CBAM output)
  - [-1, 1, C2f, [256]]                           # 24  P5/32-large
  - [[18, 21, 24], 1, Detect, [nc]]               # 25  Detect(P3, P4, P5)
"""

with open(YAML_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print(f"CBAM YAML 已生成: {YAML_PATH}")
print(open(YAML_PATH).read())

In [ ]:
# 训练 YOLOv8n + CBAM
YOLO_DATA_DIR = os.path.join(PROJECT_ROOT, "dataset", "rsna_yolo")
DATA_PATH = os.path.join(YOLO_DATA_DIR, "data.yaml")
RUNS_DIR = os.path.join(PROJECT_ROOT, "runs_rsna")

print("=" * 50)
print("Running YOLOv8n + CBAM Attention")
print("YAML:", YAML_PATH)
print("Data:", DATA_PATH)
print("Runs:", RUNS_DIR)
print("=" * 50)

# 从 YAML 构建模型，并加载 YOLOv8n 预训练权重
model = YOLO(YAML_PATH).load("yolov8n.pt")

model.train(
    data=DATA_PATH,
    epochs=80,
    imgsz=512,
    batch=8,
    device="0",
    workers=4,
    patience=20,
    project=RUNS_DIR,
    name="cbam",
    exist_ok=True,
    verbose=True,
    amp=True,
)

In [ ]:
# 评估模型
metrics = model.val()
print("=" * 50)
print("YOLOv8n + CBAM 训练完成，最终评估指标：")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Recall: {metrics.box.recall.mean():.3f}")
print("=" * 50)

In [ ]:
# 查看训练结果图
from IPython.display import Image
Image(os.path.join(RUNS_DIR, "cbam", "results.png"), width=800)